<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_02_algebraic_pinn.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 02 — Estimation With the Physics as a Residual

**Paired with L12.1 · Power Grid Stability Estimation**

**Prerequisite: notebooks 00 and 01.**

Notebook 01 showed WLS collapse when the measurements ran out. Here we add the
power flow equations to the objective and see how much of the gap they close.

### What is being optimised, and what is not

Read `StateVariables` in `problem.py` before you run anything. There is **no
neural network in this notebook**, and that is deliberate: for one grid at one
instant there is nothing to generalise over, so the honest parameterisation is
the state vector itself. That makes the comparison with WLS a comparison of
*loss functions*, not of architectures — which is the fair comparison.

The objective is

$$\mathcal{L} = \mathcal{L}_{\mathrm{meas}}
+ \lambda_{\mathrm{pf}}\,\mathcal{L}_{\mathrm{pf}}$$

Two hard constraints are built in rather than penalised, exactly as in L7.1:

* the slack angle is not a variable at all, so $\theta_1 = 0$ exactly
* magnitudes pass through a scaled sigmoid,
  $V_i = V_{\min} + \left(V_{\max}-V_{\min}\right)\sigma(\hat{V}_i)$,
  so they cannot leave a sane band

### The one knob that matters

`lam_pf` weights the physics against the meters. At zero this is least squares
on the measurements alone. Very large, and it becomes a power flow solution
that ignores your readings. The interesting behaviour is in between, and
finding it is the exercise.

---

## 0 · Setup

### A design choice worth stating: there is no neural network here

This notebook is called a physics-informed estimator and it contains no
network at all. That is deliberate, and it is the reason the notebook exists.

A physics-informed method has two separable ingredients. The first is the
**physics term in the objective**, which constrains the answer where no
measurement reaches. The second is the **flexible function** that represents
the unknown, which is usually a network.

Papers almost always introduce them together, so when the result improves it
is impossible to say which ingredient did the work. Here the unknowns are the
bus voltages and angles directly, exactly as in the weighted least squares of
notebook 01. Nothing else has changed except the objective.

So whatever improvement you measure in this notebook is attributable to the
physics term alone. Notebook 03 then adds the network, and the difference
between the two is attributable to the architecture. Two experiments, one
variable each.

That is also the honest reason a network is not always needed. If your unknown
is a handful of numbers rather than a field, a physics term on those numbers is
the whole method, and it trains in a second.


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
ref = pb.load("00_reference")
V_true, th_true = ref["V"], ref["th"]
Y = pb.build_ybus()
ms_thin = pb.thin_measurements()
z_thin = pb.synth_measurements(V_true, th_true, Y, ms_thin,
                               np.random.default_rng(3))
print(f"thin set: {len(ms_thin)} measurements, "
      f"metered buses {ms_thin.measured_buses()}")

## 1 · One run, to see it work

Start with `lam_pf = 1.0`. Watch the two loss terms separately — `pb.plot_loss`
draws them individually and `pb.algebraic_pinn` prints them at the end,
precisely so you can see which one stops improving. A falling total tells you
nothing about *which* half stopped.

One note on `lbfgs_steps`. `train_two_stage` runs that many **outer** L-BFGS
steps of up to twenty inner iterations each, so `lbfgs_steps=15` below is
roughly three hundred L-BFGS iterations. Multiply by twenty before comparing
the number with anything that takes a plain `max_iter`.

In [ ]:
V_p, th_p, hist = pb.algebraic_pinn(z_thin, ms_thin, Y, lam_pf=1.0,
                                    adam_steps=3000, lbfgs_steps=15)

res_pinn = pb.error_table(V_true, th_true, V_p, th_p,
                          ms_thin.measured_buses(), label="PINN, thin set, λ=1")
fig = pb.plot_loss(hist, "algebraic estimator"); plt.show()
fig = pb.plot_network_state(V_p, th_p, V_true, th_true,
                            ms_thin.measured_buses(), "PINN, thin set"); plt.show()

**Expected output**

> Both loss terms fall by several orders of magnitude, with L-BFGS finishing well
> below where Adam left off.
>
> The unmetered-bus error should be **substantially better than WLS's 3.0e-2** on
> the same measurements. How much better depends on λ — that is the next section,
> and if you see no improvement at all, check that you passed the *thin* set.
>
> *(These runs have not been executed by the author — torch was unavailable in
> the authoring environment. If something here fails, that is worth reporting
> rather than working around.)*

## TODO 1 — the λ sweep

Sweep `lam_pf` over several decades and plot the worst-bus error at **metered**
and **unmetered** buses separately.

Plot both. At metered buses the curve is nearly flat, because the meters
already pin those buses down — and a student who plots only that concludes λ
does not matter. The trade-off is visible only at the unmetered buses.

L9.1 made you sweep a weight and read the trade-off off a curve. Same habit.

In [ ]:
# TODO: sweep lam_pf and fill these lists, then plot with pb.plot_sweep().
#
#   lams = np.logspace(-3, 3, 13)
#   err_m, err_u = [], []
#   for lam in lams:
#       Vp, thp, _ = pb.algebraic_pinn(z_thin, ms_thin, Y, lam_pf=lam,
#                                      adam_steps=2000, lbfgs_steps=10,
#                                      verbose=False)
#       r = pb.error_table(V_true, th_true, Vp, thp,
#                          ms_thin.measured_buses(), label=f"lambda={lam:g}")
#       err_m.append(r["metered"]["V_worst"])
#       err_u.append(r["unmetered"]["V_worst"])
#   fig = pb.plot_sweep(lams, err_m, err_u, chosen=...)
#
# Then state, in a markdown cell:
#   - the λ you chose and why
#   - what happens at the extremes, and whether it matches the lecture's claim
raise NotImplementedError

## TODO 2 — the topology failure

This is the dangerous one from L12.1 slide 18, and it is worth feeling rather
than being told.

Give the estimator a **wrong** admittance matrix — build Y with a branch removed
— while the measurements still come from the true network. Then look at the
loss and at the error.

In [ ]:
# TODO:
#   Y_wrong = pb.build_ybus(outage=1)       # estimator believes a line is out
#   V_w, th_w, hist_w = pb.algebraic_pinn(z_thin, ms_thin, Y_wrong,
#                                         lam_pf=<yours>)
#
# Compare: does the loss look worse than the correct-topology run?
#          (hist_w["meas"][-1] and hist_w["pf"][-1] against hist's)
# Compare: is the state estimate worse?
#
# The point of the exercise is that the answers to those two questions differ.
# Say what a control room could do about that.
raise NotImplementedError

In [ ]:
pb.save("02_algebraic", V_pinn=V_p, th_pinn=th_p)
print("\nnotebook 02 complete — go to 03_dynamic_pinn")